# Cross-Scale SAE Validation: 16k vs 65k

Replicates the main pipeline of the paper using the 65k-feature SAE (same layer 13, Gemma 2 2B) to check whether:

1. The **LSI** produces consistent distributions across the two scales
2. The **PT-specific features** found in 16k have analogues in 65k
3. **Feature 1215** (gender) has a functional equivalent in 65k
4. **Distributed** phenomena (crase, clitics) remain distributed in 65k

**Null hypothesis:** if crase/clitics are distributed in 16k due to lack of resolution, 65k should reveal localized features. If they remain distributed → strong evidence of genuine distribution.

**Estimated runtime:** ~20 min on T4

In [ ]:
!pip install transformer_lens sae_lens datasets -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "./data/checkpoints/"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import torch, numpy as np, json, time
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
LAYER = 13
HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"

SAE_16K_ID = f"layer_{LAYER}/width_16k/canonical"
SAE_65K_ID = f"layer_{LAYER}/width_65k/canonical"
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"

SEQ_LEN = 128
BATCH_SIZE = 4
N_SAMPLES_PT = 500
N_SAMPLES_EN = 500

from sae_lens import SAE, HookedSAETransformer
print("Carregando modelo...")
model = HookedSAETransformer.from_pretrained_no_processing("gemma-2-2b", device=device, dtype=torch.float16)
tokenizer = model.tokenizer

print("Carregando SAE 65k...")
sae_65k = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_65K_ID, device=device)
print(f"SAE 65k features: {sae_65k.cfg.d_sae}")

# Only load 16k if we need comparison (try to load pre-computed stats first)
stats_16k_path = os.path.join(SAVE_DIR, "stats_layer13.pt")
HAS_16K_STATS = os.path.exists(stats_16k_path)
if HAS_16K_STATS:
    print(f"Stats 16k encontradas em {stats_16k_path}")
else:
    print("Carregando SAE 16k para comparação...")
    sae_16k = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_16K_ID, device=device)

print("Pronto.")

## Carregar corpora PT e EN

In [ ]:
from datasets import load_dataset

def collect_tokens(dataset_name, config, split, n_samples, seq_len, tokenizer):
    ds = load_dataset(dataset_name, config, split=split, streaming=True)
    all_ids = []
    for sample in ds:
        text = sample.get("text", "")
        if len(text) < 50:
            continue
        ids = tokenizer.encode(text, add_special_tokens=False)
        if len(ids) >= seq_len:
            all_ids.append(ids[:seq_len])
        if len(all_ids) >= n_samples:
            break
    return torch.tensor(all_ids)

print("Carregando tokens PT (Wikipedia)...")
tokens_pt = collect_tokens("wikimedia/wikipedia", "20231101.pt", "train", N_SAMPLES_PT, SEQ_LEN, tokenizer)
print(f"  PT: {tokens_pt.shape}")

print("Carregando tokens EN (Wikipedia)...")
tokens_en = collect_tokens("wikimedia/wikipedia", "20231101.en", "train", N_SAMPLES_EN, SEQ_LEN, tokenizer)
print(f"  EN: {tokens_en.shape}")

## Computar estatísticas 65k: frequência de ativação PT vs EN

In [ ]:
def compute_feature_stats_65k(model, sae, tokens, batch_size=BATCH_SIZE):
    n_features = sae.cfg.d_sae
    total_acts = torch.zeros(n_features, device="cpu")
    total_count = 0

    for i in tqdm(range(0, len(tokens), batch_size)):
        batch = tokens[i:i+batch_size].to(device)
        with torch.no_grad():
            _, cache = model.run_with_cache(batch, names_filter=[HOOK_NAME])
            resid = cache[HOOK_NAME]
            acts = sae.encode(resid)
            fired = (acts > 0).float().sum(dim=(0, 1))
            total_acts += fired.cpu()
            total_count += batch.shape[0] * batch.shape[1]
        del cache, resid, acts
        torch.cuda.empty_cache()

    freq = total_acts / total_count
    return freq, total_count

print("Computando stats PT (65k)...")
t0 = time.time()
freq_pt_65k, n_pt = compute_feature_stats_65k(model, sae_65k, tokens_pt)
print(f"  Concluído em {time.time()-t0:.0f}s, {n_pt} tokens")

print("Computando stats EN (65k)...")
t0 = time.time()
freq_en_65k, n_en = compute_feature_stats_65k(model, sae_65k, tokens_en)
print(f"  Concluído em {time.time()-t0:.0f}s, {n_en} tokens")

torch.save({"freq_pt": freq_pt_65k, "freq_en": freq_en_65k, "n_pt": n_pt, "n_en": n_en},
           os.path.join(SAVE_DIR, "stats_65k_layer13.pt"))
print("Stats 65k salvas.")

## Compute LSI for 65k and compare with 16k

In [ ]:
from scipy import stats as sp_stats

eps = 1e-8
lsi_65k = (freq_pt_65k - freq_en_65k) / (freq_pt_65k + freq_en_65k + eps)

pt_specific_65k = (lsi_65k > 0.3).sum().item()
en_specific_65k = (lsi_65k < -0.3).sum().item()
bilingual_65k = ((lsi_65k >= -0.3) & (lsi_65k <= 0.3)).sum().item()

n_features_65k = sae_65k.cfg.d_sae
print("="*60)
print(f"LSI Distribution (65k, layer 13)")
print("="*60)
print(f"Total features: {n_features_65k}")
print(f"PT-specific (LSI > 0.3): {pt_specific_65k} ({pt_specific_65k/n_features_65k:.1%})")
print(f"EN-specific (LSI < -0.3): {en_specific_65k} ({en_specific_65k/n_features_65k:.1%})")
print(f"Bilingual (|LSI| <= 0.3): {bilingual_65k} ({bilingual_65k/n_features_65k:.1%})")
print(f"\nMean LSI: {lsi_65k.mean().item():.4f}")
print(f"Median LSI: {lsi_65k.median().item():.4f}")
print(f"Std LSI: {lsi_65k.std().item():.4f}")

# Comparação com 16k (da Table 4 do paper)
print("\n" + "="*60)
print("Comparação 16k vs 65k")
print("="*60)
n_16k = 16384
pt_16k = 674  # from paper
en_16k = 10123  # from paper
print(f"{'Métrica':<25} {'16k':>10} {'65k':>10}")
print("-"*50)
print(f"{'PT-specific (count)':<25} {pt_16k:>10} {pt_specific_65k:>10}")
print(f"{'PT-specific (%)':<25} {pt_16k/n_16k:>9.1%} {pt_specific_65k/n_features_65k:>9.1%}")
print(f"{'EN-specific (count)':<25} {en_16k:>10} {en_specific_65k:>10}")
print(f"{'EN-specific (%)':<25} {en_16k/n_16k:>9.1%} {en_specific_65k/n_features_65k:>9.1%}")

# Ratio: o 65k "resolve" mais features PT?
ratio_pt = (pt_specific_65k / n_features_65k) / (pt_16k / n_16k)
print(f"\nRatio PT%_65k / PT%_16k: {ratio_pt:.2f}")
if ratio_pt > 1.5:
    print("→ A 65k resolve MAIS features PT (possível sub-resolução na 16k)")
elif ratio_pt < 0.7:
    print("→ 65k resolves FEWER PT features (PT features are sparser at high resolution)")
else:
    print("→ Similar proportion → consistent result across scales")

## Directed probes on 65k: test whether gender, crase, clitics have localized features

In [ ]:
PROBES = {
    "gender": {
        "positive": [
            "A menina bonita correu pelo parque.",
            "A professora dedicada ensinou a lição.",
            "A diretora nova chegou na escola.",
            "A médica experiente atendeu o paciente.",
            "A engenheira brasileira projetou a ponte.",
        ],
        "negative": [
            "O menino bonito correu pelo parque.",
            "O professor dedicado ensinou a lição.",
            "O diretor novo chegou na escola.",
            "O médico experiente atendeu o paciente.",
            "O engenheiro brasileiro projetou a ponte.",
        ],
    },
    "crase": {
        "positive": [
            "Ele foi à escola todos os dias.",
            "Ela se referiu à diretora do colégio.",
            "O aluno entregou o trabalho à professora.",
            "Nós fomos à praia no fim de semana.",
            "Ele assistiu à palestra com atenção.",
        ],
        "negative": [
            "Ele foi ao mercado todos os dias.",
            "Ela se referiu ao diretor do colégio.",
            "O aluno entregou o trabalho ao professor.",
            "Nós fomos ao parque no fim de semana.",
            "Ele assistiu ao jogo com atenção.",
        ],
    },
    "clitics": {
        "positive": [
            "Ele me disse a verdade ontem.",
            "O professor nos explicou a matéria.",
            "Ela se arrumou para a festa.",
            "Ele lhe entregou o presente.",
            "Eles nos convidaram para o jantar.",
        ],
        "negative": [
            "He told me the truth yesterday.",
            "The teacher explained the subject to us.",
            "She got ready for the party.",
            "He gave her the present.",
            "They invited us for dinner.",
        ],
    },
}

def get_feature_activations(model, sae, tokenizer, text, hook_name=HOOK_NAME):
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
    with torch.no_grad():
        _, cache = model.run_with_cache(input_ids, names_filter=[hook_name])
        resid = cache[hook_name]
        acts = sae.encode(resid)
        mean_acts = acts.mean(dim=1).squeeze(0)
    del cache, resid, acts
    torch.cuda.empty_cache()
    return mean_acts.cpu()

def find_discriminative_features(model, sae, tokenizer, positive_texts, negative_texts, top_k=20):
    """Find features that activate more for positive than negative texts."""
    pos_acts = torch.stack([get_feature_activations(model, sae, tokenizer, t) for t in positive_texts])
    neg_acts = torch.stack([get_feature_activations(model, sae, tokenizer, t) for t in negative_texts])

    mean_pos = pos_acts.mean(dim=0)
    mean_neg = neg_acts.mean(dim=0)

    diff = mean_pos - mean_neg
    ratio = mean_pos / (mean_neg + 1e-8)

    top_diff_idx = diff.argsort(descending=True)[:top_k]
    top_ratio_idx = ratio.argsort(descending=True)[:top_k]

    return {
        "top_by_diff": [(idx.item(), diff[idx].item(), mean_pos[idx].item(), mean_neg[idx].item())
                        for idx in top_diff_idx],
        "top_by_ratio": [(idx.item(), ratio[idx].item(), mean_pos[idx].item(), mean_neg[idx].item())
                         for idx in top_ratio_idx],
        "mean_pos_all": mean_pos,
        "mean_neg_all": mean_neg,
    }

print("Buscando features discriminativas na SAE 65k...")
probe_results_65k = {}
for phenomenon, texts in PROBES.items():
    print(f"\n--- {phenomenon.upper()} ---")
    result = find_discriminative_features(model, sae_65k, tokenizer,
                                          texts["positive"], texts["negative"])
    probe_results_65k[phenomenon] = result

    print(f"Top 10 features por diferença (pos - neg):")
    for fid, d, mp, mn in result["top_by_diff"][:10]:
        lsi_val = lsi_65k[fid].item()
        print(f"  Feature {fid:>6}: diff={d:+.4f}  pos={mp:.4f}  neg={mn:.4f}  LSI={lsi_val:+.3f}")

## Teste de localidade: ablação da top feature de gênero na 65k

In [ ]:
top_gender_65k = probe_results_65k["gender"]["top_by_diff"][0][0]
top_crase_65k = probe_results_65k["crase"]["top_by_diff"][0][0]
top_clitics_65k = probe_results_65k["clitics"]["top_by_diff"][0][0]

print(f"Top gender feature (65k): {top_gender_65k}")
print(f"Top crase feature (65k):  {top_crase_65k}")
print(f"Top clitics feature (65k): {top_clitics_65k}")

def generate_with_ablation(model, sae, tokenizer, prompt, feature_id, max_new=80, seed=42):
    torch.manual_seed(seed)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    fid_t = torch.tensor([feature_id], device=device)

    def hook_fn(value, hook):
        with torch.no_grad():
            sae_acts = sae.encode(value)
            modified = sae_acts.clone()
            modified[:, :, fid_t] = 0.0
            reconstructed = sae.decode(modified)
            error = value - sae.decode(sae_acts)
            return reconstructed + error

    generated = input_ids.clone()
    for _ in range(max_new):
        with torch.no_grad():
            logits = model.run_with_hooks(generated, fwd_hooks=[(HOOK_NAME, hook_fn)])
            next_logits = logits[0, -1, :] / 0.7
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
    return tokenizer.decode(generated[0], skip_special_tokens=True)

def generate_baseline(model, tokenizer, prompt, max_new=80, seed=42):
    torch.manual_seed(seed)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=max_new, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0], skip_special_tokens=True)

TEST_PROMPTS = [
    "A nova professora da escola",
    "O menino e a menina estavam",
    "A diretora apresentou o relatório",
    "Ele entregou o presente à",
]

print("\n=== Ablação do top gender feature (65k) ===")
for prompt in TEST_PROMPTS:
    base = generate_baseline(model, tokenizer, prompt)
    ablated = generate_with_ablation(model, sae_65k, tokenizer, prompt, top_gender_65k)
    print(f"\nPrompt: {prompt}")
    print(f"  Baseline: {base}")
    print(f"  Ablated:  {ablated}")

print("\n=== Ablação do top crase feature (65k) ===")
for prompt in TEST_PROMPTS:
    ablated = generate_with_ablation(model, sae_65k, tokenizer, prompt, top_crase_65k)
    print(f"\nPrompt: {prompt}")
    print(f"  Ablated:  {ablated}")

## Concentration vs Distribution: compare the spread of top features

If crase/clitics are distributed, the top-k features should have smoother diffs (low spread).
If gender is localized, the top feature should concentrate most of the signal.

In [ ]:
print("="*60)
print("Análise de localidade: concentração do sinal nos top-k features")
print("="*60)

for phenomenon in ["gender", "crase", "clitics"]:
    result = probe_results_65k[phenomenon]
    diffs = [d for _, d, _, _ in result["top_by_diff"][:20]]
    top1 = diffs[0]
    top5_sum = sum(diffs[:5])
    top10_sum = sum(diffs[:10])
    top20_sum = sum(diffs[:20])

    concentration_1_5 = top1 / (top5_sum + 1e-8)
    concentration_5_20 = top5_sum / (top20_sum + 1e-8)

    print(f"\n{phenomenon.upper()}:")
    print(f"  Top 1 diff:     {top1:.4f}")
    print(f"  Top 5 sum:      {top5_sum:.4f}")
    print(f"  Top 10 sum:     {top10_sum:.4f}")
    print(f"  Top 20 sum:     {top20_sum:.4f}")
    print(f"  Concentração top1/top5: {concentration_1_5:.1%}")
    print(f"  Concentração top5/top20: {concentration_5_20:.1%}")

    if concentration_1_5 > 0.5:
        print(f"  → LOCALIZADO: top feature domina ({concentration_1_5:.0%} do sinal)")
    elif concentration_1_5 > 0.3:
        print(f"  → SEMI-LOCALIZADO: concentração moderada")
    else:
        print(f"  → DISTRIBUÍDO: sinal espalhado entre features")

## Save results

In [ ]:
results_65k = {
    "experiment": "sae_65k_cross_scale_validation",
    "layer": LAYER,
    "n_features_65k": int(sae_65k.cfg.d_sae),
    "n_samples_pt": N_SAMPLES_PT,
    "n_samples_en": N_SAMPLES_EN,
    "seq_len": SEQ_LEN,
    "lsi_distribution_65k": {
        "pt_specific": pt_specific_65k,
        "en_specific": en_specific_65k,
        "bilingual": bilingual_65k,
        "mean_lsi": float(lsi_65k.mean()),
        "median_lsi": float(lsi_65k.median()),
        "std_lsi": float(lsi_65k.std()),
    },
    "comparison_16k": {
        "pt_specific_16k": 674,
        "pt_pct_16k": 674 / 16384,
        "pt_pct_65k": pt_specific_65k / sae_65k.cfg.d_sae,
        "ratio": ratio_pt,
    },
    "probe_results_65k": {},
    "locality_analysis": {},
}

for phenomenon in ["gender", "crase", "clitics"]:
    result = probe_results_65k[phenomenon]
    results_65k["probe_results_65k"][phenomenon] = {
        "top_by_diff": [
            {"feature_id": fid, "diff": d, "mean_pos": mp, "mean_neg": mn, "lsi": float(lsi_65k[fid])}
            for fid, d, mp, mn in result["top_by_diff"][:20]
        ],
    }
    diffs = [d for _, d, _, _ in result["top_by_diff"][:20]]
    top1 = diffs[0]
    top5_sum = sum(diffs[:5])
    top20_sum = sum(diffs[:20])
    results_65k["locality_analysis"][phenomenon] = {
        "top1_diff": top1,
        "top5_sum": top5_sum,
        "top20_sum": top20_sum,
        "concentration_top1_top5": top1 / (top5_sum + 1e-8),
        "concentration_top5_top20": top5_sum / (top20_sum + 1e-8),
    }

out_path = os.path.join(SAVE_DIR, "exp_sae_65k_results.json")
with open(out_path, "w") as f:
    json.dump(results_65k, f, indent=2, ensure_ascii=False, default=str)

local_path = "/content/exp_sae_65k_results.json"
with open(local_path, "w") as f:
    json.dump(results_65k, f, indent=2, ensure_ascii=False, default=str)

print(f"Salvo em: {out_path}")
print(f"Salvo em: {local_path}")